In [1]:
import pandas as pd
from collections import Counter

In [ ]:
df = pd.read_csv("../data/paragraph_turns_full.csv")
df = df[['sentence', 'collectiveAction', 'racialJustice', 'sentence_length']]
df.head(1)

In [ ]:
filenames = {
    "relief": "../data/annotation/processing_log_relief.txt",
    "pride": "../data/annotation/processing_log_pride.txt",
    "guilt": "../data/annotation/processing_log_guilt.txt",
    "excitement": "../data/annotation/processing_log_excitement.txt",
    "disappointment": "../data/annotation/processing_log_dissapointment.txt",  
}

label_counts = {}

for emotion, filepath in filenames.items():
    with open(filepath, "r") as f:
        labels = []
        for line in f:
            if "label=" in line:
                label = int(line.strip().split("label=")[-1])
                labels.append(label)
        label_counts[emotion] = Counter(labels)

for emotion, counts in label_counts.items():
    print(f"{emotion.capitalize()} label counts: {dict(counts)}")

In [ ]:
def extract_labels(filepath):
    labels = {}
    with open(filepath, "r") as f:
        for line in f:
            if "Row" in line and "label=" in line:
                parts = line.strip().split("label=")
                row_num = int(parts[0].split()[1].replace(":", ""))
                label = int(parts[1])
                labels[row_num] = label
    return labels

In [ ]:
emotion_labels = {emotion: extract_labels(path) for emotion, path in filenames.items()}

In [ ]:
for emotion, labels_dict in emotion_labels.items():
    df[emotion] = df.index.map(lambda i: labels_dict.get(i, 0))  

In [ ]:
df_filtered = df[df['sentence_length'] <= 10]
df_filtered = df_filtered[df_filtered['sentence_length'] > 2]
#df_collective_action = df_filtered[df_filtered['collectiveAction'] == 1]

In [ ]:
all_emotions = ['relief', 'pride', 'guilt', 'excitement', 'disappointment']
subset_df = df_filtered[df_filtered[all_emotions].any(axis=1)]
subset_df

In [ ]:
# def sample_balanced_rows(df, emotion, n_per_type=5, seed=0, exclude_indices=set()):
#     pred_1 = df[(df[emotion] == 1) & (~df.index.isin(exclude_indices))]
#     pred_0 = df[(df[emotion] == 0) & (~df.index.isin(exclude_indices))]

#     # Sample from each label category (proxy for TP, FN, FP, TN)
#     tp_like = pred_1.sample(n=n_per_type, random_state=seed)
#     fn_like = pred_0.sample(n=n_per_type, random_state=seed + 1)
#     fp_like = pred_1.sample(n=n_per_type, random_state=seed + 2)
#     tn_like = pred_0.sample(n=n_per_type, random_state=seed + 3)

#     combined = pd.concat([tp_like, fn_like, fp_like, tn_like])
#     combined = combined.drop_duplicates()
#     combined["target_emotion"] = emotion
#     return combined

In [ ]:
def sample_balanced_rows(df, emotion, seed=0, exclude_indices=set()):
    df = df[~df.index.isin(exclude_indices)]

    sampled_rows = []

    for ca_value in [0, 1]:
        subset = df[df['collectiveAction'] == ca_value]

        pred_1 = subset[subset[emotion] == 1]
        pred_0 = subset[subset[emotion] == 0]

        # For this CA value, take 5 from each predicted label
        pred_1_sample = pred_1.sample(n=5, random_state=seed + ca_value * 10 + 1)
        pred_0_sample = pred_0.sample(n=5, random_state=seed + ca_value * 10 + 2)

        sampled = pd.concat([pred_1_sample, pred_0_sample])
        sampled_rows.append(sampled)

    combined = pd.concat(sampled_rows).drop_duplicates()
    combined["target_emotion"] = emotion
    return combined

In [ ]:
all_emotions = ['relief', 'pride', 'guilt', 'excitement', 'disappointment']
final_sample = pd.DataFrame()
used_indices = set()

for i, emotion in enumerate(all_emotions):
    sample = sample_balanced_rows(
        df_filtered,
        emotion,
        seed=200 + i,
        exclude_indices=used_indices
    )
    used_indices.update(sample.index)
    final_sample = pd.concat([final_sample, sample])

In [ ]:
final_sample = final_sample.sample(frac=1, random_state=999).reset_index(drop=True)
final_sample[["sentence", "collectiveAction", "target_emotion"] + all_emotions].head(100)

In [ ]:
final_sample[["sentence", "target_emotion"] + all_emotions].to_csv("../data/emotion_annotation_full.csv", index=False)

## Evaluation

In [1]:
import pandas as pd
import krippendorff

theodora = pd.read_csv("../data/annotation/emotion_annotation_theodora.csv")
arianna = pd.read_csv("../data/annotation/emotion_annotation_arianna.csv")
davide = pd.read_csv("../data/annotation/emotion_annotation_davide.csv")

emotion_columns = ["relief", "pride", "guilt", "excitement", "disappointment"]

alpha_results = {}

for col in emotion_columns:
    annotations = [theodora[col].tolist(), arianna[col].tolist(), davide[col].tolist()]
    
    alpha = krippendorff.alpha(reliability_data=annotations, level_of_measurement='nominal')
    
    alpha_results[col] = alpha

# Display results
for emotion, alpha in alpha_results.items():
    print(f"{emotion}: Krippendorff's alpha = {alpha:.3f}")

relief: Krippendorff's alpha = 0.318
pride: Krippendorff's alpha = 0.624
guilt: Krippendorff's alpha = 0.216
excitement: Krippendorff's alpha = 0.369
disappointment: Krippendorff's alpha = 0.545


In [2]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from scipy.stats import mode

# Load annotator and gold standard data
theodora = pd.read_csv("../data/annotation/emotion_annotation_theodora.csv")
arianna = pd.read_csv("../data/annotation/emotion_annotation_arianna.csv")
davide = pd.read_csv("../data/annotation/emotion_annotation_davide.csv")
gold = pd.read_csv("../data/emotion_annotation_full.csv")

emotion_columns = ["relief", "pride", "guilt", "excitement", "disappointment"]

results = {}

for emotion in emotion_columns:
    # Stack annotations and compute majority vote (axis=0 means per item)
    annotations = pd.DataFrame({
        "theodora": theodora[emotion],
        "arianna": arianna[emotion],
        "davide": davide[emotion]
    })

    # Use scipy.stats.mode to compute majority vote
    majority_vote, _ = mode(annotations, axis=1, nan_policy='omit', keepdims=False)
    
    # Ground truth
    gold_labels = gold[emotion]

    # Filter out rows where gold is NaN or majority vote is NaN
    mask = ~pd.isnull(gold_labels) & ~pd.isnull(majority_vote)
    y_true = gold_labels[mask].astype(int)
    y_pred = majority_vote[mask].astype(int)

    # Compute metrics
    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    results[emotion] = {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn
    }

# Display results
for emotion, metrics in results.items():
    print(f"\nEmotion: {emotion}")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.3f}" if isinstance(value, float) else f"{metric}: {value}")


Emotion: relief
accuracy: 0.820
precision: 0.333
recall: 0.125
f1: 0.182
tp: 2
fp: 4
fn: 14
tn: 80

Emotion: pride
accuracy: 0.740
precision: 0.421
recall: 0.348
f1: 0.381
tp: 8
fp: 11
fn: 15
tn: 66

Emotion: guilt
accuracy: 0.880
precision: 0.333
recall: 0.091
f1: 0.143
tp: 1
fp: 2
fn: 10
tn: 87

Emotion: excitement
accuracy: 0.850
precision: 0.455
recall: 0.357
f1: 0.400
tp: 5
fp: 6
fn: 9
tn: 80

Emotion: disappointment
accuracy: 0.720
precision: 0.368
recall: 0.304
f1: 0.333
tp: 7
fp: 12
fn: 16
tn: 65


In [6]:
# import pandas as pd
# from sklearn.metrics import accuracy_score

# # Load gold and individual annotator files
# gold = pd.read_csv("../data/emotion_annotation_full.csv")
# theodora = pd.read_csv("../data/annotation/emotion_annotation_theodora.csv")
# arianna = pd.read_csv("../data/annotation/emotion_annotation_arianna.csv")

# # Emotion columns to evaluate
# emotion_columns = ["relief", "pride", "guilt", "excitement", "disappointment"]

# # Store accuracy results
# accuracy = {
#     "Theodora": {},
#     "Arianna": {}
# }

# # Loop through emotion columns and compute accuracy
# for col in emotion_columns:
#     accuracy["Theodora"][col] = accuracy_score(gold[col], theodora[col])
#     accuracy["Arianna"][col] = accuracy_score(gold[col], arianna[col])

# # Print results
# print("Accuracy Compared to Gold Labels:\n")
# for annotator, scores in accuracy.items():
#     print(f"{annotator}:")
#     for emotion, acc in scores.items():
#         print(f"  {emotion}: {acc:.3f}")
#     print()

In [5]:
# import pandas as pd
# from sklearn.metrics import (
#     accuracy_score, precision_recall_fscore_support, confusion_matrix
# )

# # Load gold and individual annotator files
# gold = pd.read_csv("../data/emotion_annotation_full.csv")
# theodora = pd.read_csv("../data/annotation/emotion_annotation_theodora.csv")
# arianna = pd.read_csv("../data/annotation/emotion_annotation_arianna.csv")

# # Emotion columns to evaluate
# emotion_columns = ["relief", "pride", "guilt", "excitement", "disappointment"]

# # Store results
# results = {
#     "Theodora": {},
#     "Arianna": {}
# }

# # Function to compute all metrics
# def compute_metrics(y_true, y_pred):
#     acc = accuracy_score(y_true, y_pred)
#     precision, recall, f1, _ = precision_recall_fscore_support(
#         y_true, y_pred, average="binary", zero_division=0
#     )
#     tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
#     return {
#         "accuracy": acc,
#         "precision": precision,
#         "recall": recall,
#         "f1": f1,
#         "tp": tp,
#         "fp": fp,
#         "fn": fn,
#         "tn": tn
#     }

# # Loop through emotion columns and compute metrics
# for col in emotion_columns:
#     results["Theodora"][col] = compute_metrics(gold[col], theodora[col])
#     results["Arianna"][col] = compute_metrics(gold[col], arianna[col])

# # Print results
# for annotator, emotions in results.items():
#     print(f"{annotator}:\n")
#     for emotion, metrics in emotions.items():
#         print(f"{emotion}:")
#         for metric, value in metrics.items():
#             print(f"  {metric}: {value:.3f}" if isinstance(value, float) else f"  {metric}: {value}")
#         print()

In [4]:
# import pandas as pd
# from sklearn.metrics import (
#     accuracy_score, precision_recall_fscore_support
# )

# # Load gold and individual annotator files
# gold = pd.read_csv("../data/emotion_annotation_full.csv")
# theodora = pd.read_csv("../data/annotation/emotion_annotation_theodora.csv")
# arianna = pd.read_csv("../data/annotation/emotion_annotation_arianna.csv")

# # Emotion columns to evaluate
# emotion_columns = ["relief", "pride", "guilt", "excitement", "disappointment"]

# # Function to compute metrics
# def compute_metrics(y_true, y_pred):
#     acc = accuracy_score(y_true, y_pred)
#     precision, recall, f1, _ = precision_recall_fscore_support(
#         y_true, y_pred, average="binary", zero_division=0
#     )
#     return acc, precision, recall, f1

# # Collect rows for the summary table
# rows = []

# for col in emotion_columns:
#     acc, prec, rec, f1 = compute_metrics(gold[col], theodora[col])
#     rows.append(["Theodora", col, acc, prec, rec, f1])
    
#     acc, prec, rec, f1 = compute_metrics(gold[col], arianna[col])
#     rows.append(["Arianna", col, acc, prec, rec, f1])

# # Create the summary table
# df_results = pd.DataFrame(rows, columns=[
#     "Annotator", "Emotion", "Accuracy", "Precision", "Recall", "F1"
# ])

# # Round for display
# print(df_results.round(3))